In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# 1. Load the Datasets
train_df = pd.read_csv('/kaggle/input/datasets/ranumishra29/smitmish-titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/ranumishra29/smitmish-titanic/test.csv')

passenger_ids = test_df['PassengerId']

# 2. Robust Cleaning Function
def preprocess_titanic(df):
    df = df.copy()
    
    # Fill numeric missing values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # Fill categorical missing values
    df['Embarked'] = df['Embarked'].fillna('S')
    
    # Map text values to numbers
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1}).fillna(0)
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)
    
    # Drop non-numeric / unneeded columns
    cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    # Safety catch: Fill any lingering NaNs across all remaining columns
    df = df.fillna(0)
    
    return df

# Apply Preprocessing
X_train = preprocess_titanic(train_df)
X_test = preprocess_titanic(test_df)

# Separate target column (Survived) from training set
y_train = X_train.pop('Survived')

# 3. Quick Verification check before fitting
print("Any NaNs left in X_train?:", X_train.isna().sum().sum())
print("Data Types:\n", X_train.dtypes)

# 4. Fit the Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Make Predictions & Export CSV
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)
print("\nSuccess! 'submission.csv' generated cleanly.")

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv
/kaggle/input/datasets/ranumishra29/smitmish-titanic/train.csv
/kaggle/input/datasets/ranumishra29/smitmish-titanic/test.csv
Any NaNs left in X_train?: 0
Data Types:
 Pclass        int64
Sex           int64
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked      int64
dtype: object

Success! 'submission.csv' generated cleanly.
